# Radar Detection Pipeline
## Data Setup

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Class mapping based on your requirements
CLASS_MAP = {
    1: 'Ship',
    2: 'Noise',
    3: 'Unknown TGT',
    4: 'Coast/Port',
    5: 'My Ship',
    6: 'AIS Lost'
}

def analyze_dataset(data_dir):
    class_counts = Counter()
    anno_types = Counter()
    
    for filename in os.listdir(data_dir):
        if filename.endswith(".json"):
            with open(os.path.join(data_dir, filename), 'r') as f:
                data = json.load(f)
                
            for anno in data.get('annotations', []):
                cat_id = anno.get('category_id')
                anno_type = anno.get('type')
                
                if cat_id in CLASS_MAP:
                    class_counts[CLASS_MAP[cat_id]] += 1
                anno_types[anno_type] += 1

    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Class Distribution
    sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), ax=axes[0], palette="viridis")
    axes[0].set_title('Class Distribution in W-Radar Dataset')
    axes[0].set_ylabel('Number of Annotations')
    
    # Annotation Type Distribution
    axes[1].pie(anno_types.values(), labels=anno_types.keys(), autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
    axes[1].set_title('Annotation Types (BBox vs Segmentation)')
    
    plt.tight_layout()
    plt.savefig(f'dataset_analysis_{os.path.basename(data_dir)}.png')
    plt.show()

In [ ]:
dataset_folders = [
    'RealImages',
    'IMMAGINI/Cartour delta 00.30/202109150030',
    'IMMAGINI/Cartour delta 1.00/202109150100',
    'IMMAGINI/Cartour delta 1.30/202109150130'
]

## Data Analysis

In [ ]:
for folder in dataset_folders:
    print(f"Analyzing dataset: {folder}")
    analyze_dataset(folder)

## Dataset Merge and Split in Folds

In [ ]:
import os
import json
import glob
from sklearn.model_selection import train_test_split, KFold

def merge_and_split_datasets(dataset_folders, output_dir, test_size=0.2, n_splits=5):
    os.makedirs(output_dir, exist_ok=True)
    all_files = []
    
    # 1. Gather ALL files across ALL folders
    for folder in dataset_folders:
        json_files = glob.glob(os.path.join(folder, '*.json'))
        for jf in json_files:
            # We store the full relative path without extension (e.g., "RealImages/R_311782")
            base_path = os.path.splitext(jf)[0]
            all_files.append(base_path)
            
    # 2. Hold-out Test Set Split
    trainval_files, test_files = train_test_split(all_files, test_size=test_size, random_state=42)
    
    splits = {
        "test": test_files,
        "folds": {}
    }
    
    # 3. 5-Fold Cross Validation on the TrainVal portion
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(trainval_files)):
        splits["folds"][f"fold_{fold+1}"] = {
            "train": [trainval_files[i] for i in train_idx],
            "val": [trainval_files[i] for i in val_idx]
        }
        
    out_path = os.path.join(output_dir, 'merged_dataset_splits.json')
    with open(out_path, 'w') as f:
        json.dump(splits, f, indent=4)
        
    print(f"Merge & Split complete! Saved to {out_path}")
    print(f"Total Images: {len(all_files)} | TrainVal: {len(trainval_files)} | Test: {len(test_files)}")

# Execute it once:
dataset_folders = [
    'RealImages',
    'IMMAGINI/Cartour delta 00.30/202109150030',
    'IMMAGINI/Cartour delta 1.00/202109150100',
    'IMMAGINI/Cartour delta 1.30/202109150130'
]
merge_and_split_datasets(dataset_folders, 'MasterSplits')

## Dataset and Training Loop Setup

In [ ]:
import os
import json
import torch
import cv2
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor
from torch.amp import autocast, GradScaler

class WRadarDataset(Dataset):
    def __init__(self, file_list, processor, mask_circle=True):
        self.file_list = file_list
        self.processor = processor
        self.mask_circle = mask_circle
        
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        base_path = self.file_list[idx]
        img_path = f"{base_path}.png"
        json_path = f"{base_path}.json"
        
        # 1. Load Image & Convert to Grayscale -> RGB (Drops color info, keeps 3 channels)
        image = cv2.imread(img_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        
        h, w, _ = image.shape
        
        with open(json_path, 'r') as f:
            anno_data = json.load(f)
            
        instance_seg = np.zeros((h, w), dtype=np.int32)
        instance_id_to_semantic_id = {}
        
        for i, ann in enumerate(anno_data.get('annotations', [])):
            instance_id = i + 1 
            cat_id = ann['category_id']
            instance_id_to_semantic_id[instance_id] = cat_id
            
            if ann['type'] == 'bbox':
                xmin, ymin = int(ann['xmin']), int(ann['ymin'])
                xmax, ymax = xmin + int(ann['width']), ymin + int(ann['height'])
                instance_seg[ymin:ymax, xmin:xmax] = instance_id
            elif ann['type'] == 'segmentation':
                poly = np.array(ann['segmentation'][0], dtype=np.int32).reshape((-1, 2))
                cv2.fillPoly(instance_seg, [poly], instance_id)
                
        # 2. DYNAMIC CROP: The radar is a square on the left. Width always equals Height.
        crop_size = h 
        
        if w > crop_size:
            image = image[:, :crop_size, :]
            instance_seg = instance_seg[:, :crop_size]
            
            # Clean up instances that were completely cropped out into the UI area
            active_instances = np.unique(instance_seg)
            instance_id_to_semantic_id = {
                inst_id: sem_id 
                for inst_id, sem_id in instance_id_to_semantic_id.items() 
                if inst_id in active_instances
            }
            
        # 3. Apply Circular Mask (Dynamically scaled to the new square)
        if self.mask_circle:
            cur_h, cur_w = image.shape[:2]
            circle_mask = np.zeros((cur_h, cur_w), dtype=np.uint8)
            
            # Center is exactly the middle of the dynamic square
            center_x, center_y = cur_w // 2, cur_h // 2
            radius = min(center_x, center_y)
            
            # Draw a solid white circle on the black mask
            cv2.circle(circle_mask, (center_x, center_y), radius, 1, thickness=-1)
            
            # Multiply image/seg arrays by the mask to zero out the corners
            image = image * circle_mask[:, :, np.newaxis]
            instance_seg = instance_seg * circle_mask
                
        # 4. Process for Mask2Former
        inputs = self.processor(
            images=image, 
            segmentation_maps=instance_seg, 
            instance_id_to_semantic_id=instance_id_to_semantic_id,
            return_tensors="pt"
        )
        
        processed_inputs = {}
        for k, v in inputs.items():
            if isinstance(v, list):
                processed_inputs[k] = v[0]  
            elif isinstance(v, torch.Tensor):
                processed_inputs[k] = v.squeeze(0) 
            else:
                processed_inputs[k] = v
                
        return processed_inputs

# Batch IoU calculation function (Foreground vs Background)
def compute_fast_batch_iou(outputs, mask_labels):
    """Calculates a fast binary IoU (Foreground vs Background)"""
    with torch.no_grad():
        pred_masks = outputs.masks_queries_logits > 0 
        total_iou = 0.0
        
        for i in range(len(mask_labels)):
            if len(mask_labels[i]) == 0:
                continue # Skip if no ground truth in this image
                
            true_fg = (mask_labels[i].sum(dim=0) > 0).float()
            pred_resized = torch.nn.functional.interpolate(
                pred_masks[i].unsqueeze(0).float(), 
                size=true_fg.shape[-2:], 
                mode="bilinear", 
                align_corners=False
            ).squeeze(0)
            
            pred_fg = (pred_resized.sum(dim=0) > 0).float()
            
            intersection = (pred_fg * true_fg).sum()
            union = pred_fg.sum() + true_fg.sum() - intersection
            
            iou = (intersection / (union + 1e-6)).item()
            total_iou += iou
            
        return total_iou / len(mask_labels) if len(mask_labels) > 0 else 0.0

# Main Training Loop
def train_vit_segmentation(split_json_path, fold, batch_size, learning_rate, epochs, model_checkpoint):
    
    with open(split_json_path, 'r') as f:
        splits = json.load(f)
        
    train_files = splits["folds"][fold]["train"]
    val_files = splits["folds"][fold]["val"] 

    processor = Mask2FormerImageProcessor(ignore_index=0, size={"height": 512, "width": 512})
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_checkpoint)
    
    # Initialize Datasets with the circular mask (no explicit crop width needed!)
    train_dataset = WRadarDataset(train_files, processor, mask_circle=True)
    val_dataset = WRadarDataset(val_files, processor, mask_circle=True)
    
    def collate_fn(batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        pixel_mask = torch.stack([item["pixel_mask"] for item in batch]) if "pixel_mask" in batch[0] else None
        class_labels = [item["class_labels"] for item in batch]
        mask_labels = [item["mask_labels"] for item in batch]
        return {
            "pixel_values": pixel_values,
            "pixel_mask": pixel_mask,
            "class_labels": class_labels,
            "mask_labels": mask_labels
        }

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    # Ensure we are using CUDA
    device = torch.device("cuda")
    model.to(device)
    
    # Initialize the AMP Scaler
    scaler = GradScaler('cuda')
    
    print(f"Starting training on device: {device} with AMP Enabled")
    print(f"Train Size: {len(train_dataset)} | Val Size: {len(val_dataset)} | Fold: {fold}")
    
    best_val_iou = 0.0
    session_date = datetime.now().strftime("%Y%m%d_%H%M%S")
    best_model_filename = f"vit_wradar_{fold}_best_{session_date}.pth"
    
    for epoch in range(epochs):
        # ==========================================
        # TRAINING PHASE
        # ==========================================
        model.train()
        running_loss = 0.0
        running_iou = 0.0
        
        with tqdm(total=len(train_dataset), desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="img") as pbar:
            for step, batch in enumerate(train_dataloader, 1):
                optimizer.zero_grad()
                
                current_batch_size = len(batch["pixel_values"])
                pixel_values = batch["pixel_values"].to(device)
                pixel_mask = batch["pixel_mask"].to(device) if batch["pixel_mask"] is not None else None
                
                device_class_labels = [l.to(device) for l in batch["class_labels"]]
                device_mask_labels = [m.to(device) for m in batch["mask_labels"]]
                
                # --- Run the forward pass in Mixed Precision ---
                with autocast('cuda'):
                    outputs = model(
                        pixel_values=pixel_values, 
                        pixel_mask=pixel_mask,
                        class_labels=device_class_labels,
                        mask_labels=device_mask_labels
                    )
                    loss = outputs.loss
                
                # NaN Loss Check (Forward Pass)
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"\n[Warning] NaN loss at step {step}. Skipping.")
                    optimizer.zero_grad()
                    pbar.update(current_batch_size)
                    continue
                
                # --- Scale the loss and run backward pass ---
                scaler.scale(loss).backward()
                
                # Unscale the gradients so we can inspect/clip them
                scaler.unscale_(optimizer)
                
                # NaN Gradient Check (Backward Pass)
                has_corrupted_grad = False
                for param in model.parameters():
                    if param.grad is not None and not torch.isfinite(param.grad).all():
                        has_corrupted_grad = True
                        break
                        
                if has_corrupted_grad:
                    print(f"\n[Warning] Corrupted gradients at step {step}. Scaler will auto-adjust.")
                    # Notice we DO NOT use `continue` here! 
                    # We just skip clipping so we don't do math on Infinity/NaN.
                else:
                    # Clip gradients safely
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.01)

                # Step the optimizer and update the scaler
                # If grads contained NaN, scaler.step() automatically skips the optimizer update.
                scaler.step(optimizer)
                
                # scaler.update() resets the unscale state. 
                # If it saw NaNs, it dramatically lowers the scale factor for the next batch!
                scaler.update()
                
                # ... proceed to metric calculation ...
                batch_iou = compute_fast_batch_iou(outputs, device_mask_labels)
                running_loss += loss.item()
                running_iou += batch_iou
                
                pbar.update(current_batch_size)
                pbar.set_postfix({
                    "Avg Loss": f"{running_loss/step:.4f}", 
                    "Avg IoU": f"{running_iou/step:.4f}"
                })
                
        # ==========================================
        # VALIDATION PHASE
        # ==========================================
        model.eval()
        val_running_iou = 0.0
        
        with torch.no_grad():
            with tqdm(total=len(val_dataset), desc=f"Epoch {epoch+1}/{epochs} [Val]  ", unit="img") as pbar:
                for step, batch in enumerate(val_dataloader, 1):
                    current_batch_size = len(batch["pixel_values"])
                    pixel_values = batch["pixel_values"].to(device)
                    device_mask_labels = [m.to(device) for m in batch["mask_labels"]]
                    
                    outputs = model(pixel_values=pixel_values)
                    
                    batch_iou = compute_fast_batch_iou(outputs, device_mask_labels)
                    val_running_iou += batch_iou
                    
                    pbar.update(current_batch_size)
                    pbar.set_postfix({"Avg IoU": f"{val_running_iou/step:.4f}"})
                    
        epoch_val_iou = val_running_iou / len(val_dataloader)
        
        # ==========================================
        # SAVE BEST MODEL CHECK
        # ==========================================
        if epoch_val_iou > best_val_iou:
            print(f"--> Validation IoU improved from {best_val_iou:.4f} to {epoch_val_iou:.4f}! Saving model...")
            best_val_iou = epoch_val_iou
            torch.save(model.state_dict(), best_model_filename)
        else:
            print(f"--> Validation IoU did not improve. (Best: {best_val_iou:.4f})")
            
        print("-" * 50)

    print(f"Training Complete! Best model saved as: {best_model_filename} with Val IoU: {best_val_iou:.4f}")
    
    # ADD THIS LINE:
    return best_val_iou, best_model_filename

## Models Training (Swin Transformer in 5-Fold Cross Validation)

In [ ]:
# Point this to the new merged JSON we generated in the previous step
SPLIT_JSON_PATH = 'MasterSplits/merged_dataset_splits.json'

BATCH_SIZE = 1
LEARNING_RATE = 1e-5
EPOCHS = 20
MODEL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"

fold_scores = []
saved_models = []

print("==================================================")
print("STARTING 5-FOLD CROSS VALIDATION")
print("==================================================")

for fold_num in range(1, 6):
    fold_name = f"fold_{fold_num}"
    print(f"\n---> Launching {fold_name}...")
    
    best_iou, model_name = train_vit_segmentation(
        split_json_path=SPLIT_JSON_PATH, 
        fold=fold_name, 
        batch_size=BATCH_SIZE, 
        learning_rate=LEARNING_RATE, 
        epochs=EPOCHS, 
        model_checkpoint=MODEL_CHECKPOINT
    )
    
    fold_scores.append(best_iou)
    saved_models.append(model_name)

# --- Calculate Mean and Standard Deviation ---
mean_iou = np.mean(fold_scores)
std_iou = np.std(fold_scores)

print("\n==================================================")
print("CROSS VALIDATION RESULTS")
print("==================================================")
for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.4f}")
print(f"\nFinal Expected Performance (mIoU): {mean_iou:.4f} ± {std_iou:.4f}")
print("==================================================")

## Model Training with All Folds

In [ ]:
def train_final_production_model(split_json_path, batch_size, learning_rate, epochs, model_checkpoint):
    with open(split_json_path, 'r') as f:
        splits = json.load(f)
        
    # Combine fold_1's train and val to get the ENTIRE TrainVal dataset!
    full_trainval_files = splits["folds"]["fold_1"]["train"] + splits["folds"]["fold_1"]["val"]
    
    processor = Mask2FormerImageProcessor(ignore_index=0, size={"height": 512, "width": 512})
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_checkpoint)
    
    # [CHANGED]: Removed crop_width, added mask_circle=True
    train_dataset = WRadarDataset(full_trainval_files, processor, mask_circle=True)
    
    def collate_fn(batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        pixel_mask = torch.stack([item["pixel_mask"] for item in batch]) if "pixel_mask" in batch[0] else None
        class_labels = [item["class_labels"] for item in batch]
        mask_labels = [item["mask_labels"] for item in batch]
        return {
            "pixel_values": pixel_values,
            "pixel_mask": pixel_mask,
            "class_labels": class_labels,
            "mask_labels": mask_labels
        }

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    device = torch.device("cuda")
    model.to(device)
    scaler = GradScaler('cuda')
    
    print(f"Training Final Production Model on {len(train_dataset)} images...")
    final_model_filename = "vit_wradar_FINAL_PRODUCTION.pth"
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        with tqdm(total=len(train_dataset), desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="img") as pbar:
            for step, batch in enumerate(train_dataloader, 1):
                optimizer.zero_grad()
                current_batch_size = len(batch["pixel_values"])
                
                # Move to device
                pixel_values = batch["pixel_values"].to(device)
                pixel_mask = batch["pixel_mask"].to(device) if batch["pixel_mask"] is not None else None
                device_class_labels = [l.to(device) for l in batch["class_labels"]]
                device_mask_labels = [m.to(device) for m in batch["mask_labels"]]
                
                with autocast('cuda'):
                    outputs = model(
                        pixel_values=pixel_values, 
                        pixel_mask=pixel_mask,
                        class_labels=device_class_labels,
                        mask_labels=device_mask_labels
                    )
                    loss = outputs.loss
                
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad()
                    pbar.update(current_batch_size)
                    continue
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.01)
                
                scaler.step(optimizer)
                scaler.update()
                
                running_loss += loss.item()
                pbar.update(current_batch_size)
                pbar.set_postfix({"Avg Loss": f"{running_loss/step:.4f}"})
                
    # Save the final model at the end of all epochs
    torch.save(model.state_dict(), final_model_filename)
    print(f"Final Model Saved: {final_model_filename}")

In [ ]:
# ==========================================
# FINAL PRODUCTION MODEL TRAINING
# ==========================================
SPLIT_JSON_PATH = 'MasterSplits/merged_dataset_splits.json'

BATCH_SIZE = 1
LEARNING_RATE = 1e-5
EPOCHS = 20
MODEL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"

print("==================================================")
print("STARTING FINAL PRODUCTION RUN")
print("==================================================")

train_final_production_model(
    split_json_path=SPLIT_JSON_PATH, 
    batch_size=BATCH_SIZE, 
    learning_rate=LEARNING_RATE, 
    epochs=EPOCHS, 
    model_checkpoint=MODEL_CHECKPOINT
)

## Model Evaluation and Inference

In [ ]:
import os
import json
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

# ==========================================
# 1. EVALUATION CONFIGURATION
# ==========================================
SPLIT_JSON_PATH = 'MasterSplits/merged_dataset_splits.json'
BEST_MODEL_WEIGHTS = "vit_wradar_FINAL_PRODUCTION.pth" # Update to your trained weights!
MODEL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
OUTPUT_VIS_DIR = "Inference_Results"

# Select which mapping you want to test here: 'original', 'mapping_1', or 'mapping_2'
EVAL_MAPPING = 'mapping_1'

# Define the logical mappings
MAPPINGS = {
    'original': {
        'desc': 'Original Classes',
        'classes': {1: 'Ship', 2: 'Noise', 3: 'Unknown TGT', 4: 'Coast/Port', 5: 'My Ship', 6: 'AIS Lost'},
        'func': lambda x: x # No change
    },
    'mapping_1': {
        'desc': 'Mapping 1 (Unknown & Lost -> Ship)',
        'classes': {1: 'Ship (Merged)', 2: 'Noise', 4: 'Coast/Port', 5: 'My Ship'},
        'func': lambda x: np.where(np.isin(x, [3, 6]), 1, x)
    },
    'mapping_2': {
        'desc': 'Mapping 2 (Lost -> Ship)',
        'classes': {1: 'Ship (Merged)', 2: 'Noise', 3: 'Unknown TGT', 4: 'Coast/Port', 5: 'My Ship'},
        'func': lambda x: np.where(x == 6, 1, x)
    }
}

# ==========================================
# 2. MAIN EVALUATION FUNCTION
# ==========================================
def evaluate_and_visualize(mapping_key):
    os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)
    map_config = MAPPINGS[mapping_key]
    map_func = map_config['func']
    class_map = map_config['classes']
    
    print(f"--- Running Evaluation: {map_config['desc']} ---")
    
    with open(SPLIT_JSON_PATH, 'r') as f:
        splits = json.load(f)
    test_files = splits["test"]
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = Mask2FormerImageProcessor(ignore_index=0, size={"height": 512, "width": 512})
    model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_CHECKPOINT)
    model.load_state_dict(torch.load(BEST_MODEL_WEIGHTS, map_location=device))
    model.to(device)
    model.eval()
    
    # Tracking metrics: True Positives, False Positives, False Negatives
    metrics = {cid: {'TP': 0, 'FP': 0, 'FN': 0} for cid in class_map.keys()}
    
    # Store results for visualization: (mIoU, base_name, image, true_mask, pred_mask)
    image_results = []
    
    with torch.no_grad():
        for base_name in tqdm(test_files, desc="Evaluating Test Set"):
            # Because merged_dataset_splits holds relative paths like "RealImages/R_123"
            img_path = f"{base_name}.png"
            json_path = f"{base_name}.json"
            
            if not os.path.exists(img_path): continue
                
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            h, w, _ = image.shape
            
            # --- Load Ground Truth ---
            with open(json_path, 'r') as f:
                anno_data = json.load(f)
                
            true_semantic_mask = np.zeros((h, w), dtype=np.uint8)
            for ann in anno_data.get('annotations', []):
                cat_id = ann['category_id']
                if ann['type'] == 'bbox':
                    xmin, ymin = int(ann['xmin']), int(ann['ymin'])
                    xmax, ymax = xmin + int(ann['width']), ymin + int(ann['height'])
                    true_semantic_mask[ymin:ymax, xmin:xmax] = cat_id
                elif ann['type'] == 'segmentation':
                    poly = np.array(ann['segmentation'][0], dtype=np.int32).reshape((-1, 2))
                    cv2.fillPoly(true_semantic_mask, [poly], cat_id)
            
            # --- Dynamic Crop & Mask ---
            crop_size = h
            if w > crop_size:
                image = image[:, :crop_size, :]
                true_semantic_mask = true_semantic_mask[:, :crop_size]
                
            circle_mask = np.zeros((crop_size, crop_size), dtype=np.uint8)
            cv2.circle(circle_mask, (crop_size//2, crop_size//2), crop_size//2, 1, thickness=-1)
            image = image * circle_mask[:, :, np.newaxis]
            true_semantic_mask = true_semantic_mask * circle_mask
            
            # --- Forward Pass ---
            inputs = processor(images=image, return_tensors="pt")
            outputs = model(pixel_values=inputs["pixel_values"].to(device))
            
            # --- Post-Process Predictions ---
            pred_maps = processor.post_process_instance_segmentation(
                outputs, target_sizes=[(crop_size, crop_size)]
            )[0]
            
            pred_instance_mask = pred_maps['segmentation'].cpu().numpy()
            pred_semantic_mask = np.zeros((crop_size, crop_size), dtype=np.uint8)
            for seg in pred_maps['segments_info']:
                pred_semantic_mask[pred_instance_mask == seg['id']] = seg['label_id']
                
            pred_semantic_mask = pred_semantic_mask * circle_mask
            
            # --- APPLY CLASS MAPPING ---
            true_remapped = map_func(true_semantic_mask)
            pred_remapped = map_func(pred_semantic_mask)
            
            # --- Calculate Per-Image Metrics ---
            img_ious = []
            for cid in class_map.keys():
                gt_c = (true_remapped == cid)
                pr_c = (pred_remapped == cid)
                
                tp = np.sum(gt_c & pr_c)
                fp = np.sum(pr_c & ~gt_c)
                fn = np.sum(~pr_c & gt_c)
                
                metrics[cid]['TP'] += tp
                metrics[cid]['FP'] += fp
                metrics[cid]['FN'] += fn
                
                union = tp + fp + fn
                if union > 0:
                    img_ious.append(tp / union)
                    
            img_mIoU = np.mean(img_ious) if img_ious else 0.0
            image_results.append((img_mIoU, os.path.basename(base_name), image, true_remapped, pred_remapped))
            
    # ==========================================
    # 3. PRINT GLOBAL METRICS
    # ==========================================
    print("\n" + "="*60)
    print(f"GLOBAL METRICS: {map_config['desc']}")
    print("="*60)
    print(f"{'Class Name':<20} | {'IoU':<8} | {'Precision':<9} | {'Recall':<8}")
    print("-" * 60)
    
    global_ious = []
    for cid, name in class_map.items():
        tp = metrics[cid]['TP']
        fp = metrics[cid]['FP']
        fn = metrics[cid]['FN']
        
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        if (tp + fp + fn) > 0:
            global_ious.append(iou)
            print(f"{name:<20} | {iou:.4f}   | {precision:.4f}    | {recall:.4f}")
        else:
            print(f"{name:<20} | N/A      | N/A       | N/A")
            
    print("-" * 60)
    print(f"** Mean IoU (mIoU): {np.mean(global_ious):.4f} **\n")

    # ==========================================
    # 4. VISUALIZE BEST & WORST CASES
    # ==========================================
    image_results.sort(key=lambda x: x[0], reverse=True) # Sort by mIoU descending
    
    # Take top 3 and bottom 3
    cases_to_plot = []
    cases_to_plot = [("RESULT", x) for x in image_results]
        
    cmap = plt.get_cmap('nipy_spectral')
    norm = plt.Normalize(vmin=1, vmax=6)
    legend_patches = [mpatches.Patch(color=cmap(norm(cid)), label=name, alpha=0.7) for cid, name in class_map.items()]

    for rank_type, (iou, name, img, true_mask, pred_mask) in cases_to_plot:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f"[{rank_type}] {name} | Image mIoU: {iou:.4f}", fontsize=16, fontweight='bold')
        
        axes[0].imshow(img)
        axes[0].set_title("Original Radar")
        axes[0].axis('off')
        
        axes[1].imshow(img)
        axes[1].imshow(np.ma.masked_where(true_mask == 0, true_mask), cmap=cmap, norm=norm, alpha=0.7)
        axes[1].set_title("Ground Truth (Mapped)")
        axes[1].axis('off')
        
        axes[2].imshow(img)
        axes[2].imshow(np.ma.masked_where(pred_mask == 0, pred_mask), cmap=cmap, norm=norm, alpha=0.7)
        axes[2].set_title("Model Prediction")
        axes[2].axis('off')
        axes[2].legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.05, 1), title="Classes")
        
        plt.tight_layout()
        save_name = f"{OUTPUT_VIS_DIR}/{mapping_key}_{rank_type}_{name}.png"
        plt.savefig(save_name, bbox_inches='tight')
        plt.close()
        
    print(f"Visualizations saved to {OUTPUT_VIS_DIR}/")

# EXECUTE THE SCRIPT
#evaluate_and_visualize(EVAL_MAPPING)

In [ ]:
for mapping in MAPPINGS.keys():
    evaluate_and_visualize(mapping)

## YOLO Dataset

In [ ]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm

def build_yolo_bbox_dataset(split_json_path, output_dir="YOLO_Dataset"):
    """
    Converts the hybrid radar dataset into a pure bounding box dataset.
    Crops the images dynamically, applies a circular mask, removes segmentation arrays, 
    and organizes everything into YOLOv26 training format using the existing splits.
    """
    print(f"Initializing YOLO BBox Dataset creation at '{output_dir}'...")
    
    # 1. Create YOLO Directory Structure
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_dir, f"images/{split}"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, f"labels/{split}"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, f"cleaned_jsons/{split}"), exist_ok=True)

    # 2. Load the Master Splits
    with open(split_json_path, 'r') as f:
        splits = json.load(f)
        
    # We use Fold 1 to define our Train/Val split, and retain the global test set.
    dataset_splits = {
        'train': splits["folds"]["fold_1"]["train"],
        'val': splits["folds"]["fold_1"]["val"],
        'test': splits["test"]
    }

    # 3. Process Every Image
    for split_name, base_paths in dataset_splits.items():
        print(f"\nProcessing {split_name} split ({len(base_paths)} images)...")
        
        for base_path in tqdm(base_paths):
            img_path = f"{base_path}.png"
            json_path = f"{base_path}.json"
            
            if not os.path.exists(img_path) or not os.path.exists(json_path):
                print(f"Warning: Missing files for {base_path}. Skipping.")
                continue

            # Extract just the filename (e.g., "R_120282") for saving
            filename = os.path.basename(base_path)

            # --- IMAGE PROCESSING ---
            image = cv2.imread(img_path)
            h, w, _ = image.shape
            crop_size = h  # Radar is a square on the left, so width = height
            
            if w > crop_size:
                image = image[:, :crop_size, :]
                
            # Apply Circular Mask
            circle_mask = np.zeros((crop_size, crop_size), dtype=np.uint8)
            center = crop_size // 2
            cv2.circle(circle_mask, (center, center), center, 1, thickness=-1)
            image = image * circle_mask[:, :, np.newaxis] # Apply mask to all 3 channels
            
            # Save cropped & masked image
            cv2.imwrite(os.path.join(output_dir, f"images/{split_name}/{filename}.png"), image)

            # --- JSON & ANNOTATION PROCESSING ---
            with open(json_path, 'r') as f:
                data = json.load(f)

            data['image']['width'] = crop_size
            data['image']['height'] = crop_size

            yolo_labels = []
            cleaned_annotations = []

            for ann in data.get('annotations', []):
                # If it's a polygon, mathematically convert it to a Bounding Box
                if ann.get('type') == 'segmentation' and 'segmentation' in ann:
                    pts = np.array(ann['segmentation'][0]).reshape(-1, 2)
                    xmin = float(np.min(pts[:, 0]))
                    ymin = float(np.min(pts[:, 1]))
                    xmax = float(np.max(pts[:, 0]))
                    ymax = float(np.max(pts[:, 1]))
                    
                    ann['type'] = 'bbox'
                    ann['xmin'] = xmin
                    ann['ymin'] = ymin
                    ann['width'] = xmax - xmin
                    ann['height'] = ymax - ymin
                    del ann['segmentation'] # Strip the heavy mask array completely

                # Ensure BBox is inside the cropped square
                xmin = ann['xmin']
                ymin = ann['ymin']
                xmax = xmin + ann['width']
                ymax = ymin + ann['height']

                # If the box starts outside the radar square (e.g., in the UI panel), discard it
                if xmin >= crop_size:
                    continue  

                # Truncate if partially outside the crop
                xmax = min(xmax, crop_size)
                ymax = min(ymax, crop_size)
                ann['width'] = xmax - xmin
                ann['height'] = ymax - ymin
                cleaned_annotations.append(ann)

                # --- GENERATE YOLO .TXT FORMAT ---
                # YOLO uses 0-indexed classes. Our JSON classes are 1-6, so we subtract 1.
                cat_id = ann['category_id'] - 1 

                # YOLO requires normalized center coordinates (0.0 to 1.0)
                x_center = (xmin + (ann['width'] / 2.0)) / crop_size
                y_center = (ymin + (ann['height'] / 2.0)) / crop_size
                norm_w = ann['width'] / crop_size
                norm_h = ann['height'] / crop_size

                # Format: class_id x_center y_center width height
                yolo_labels.append(f"{cat_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

            data['annotations'] = cleaned_annotations

            # Save Cleaned BBox-Only JSON
            with open(os.path.join(output_dir, f"cleaned_jsons/{split_name}/{filename}.json"), 'w') as out_f:
                json.dump(data, out_f, indent=4)

            # Save YOLO Text file
            with open(os.path.join(output_dir, f"labels/{split_name}/{filename}.txt"), 'w') as out_f:
                out_f.write("\n".join(yolo_labels))

    # 4. Generate YOLO data.yaml configuration file
    yaml_content = f"""path: ../{output_dir}
train: images/train
val: images/val
test: images/test

# Classes (0-indexed for YOLO)
names:
  0: Ship
  1: Noise
  2: Unknown TGT
  3: Coast/Port
  4: My Ship
  5: AIS Lost
"""
    with open(os.path.join(output_dir, "data.yaml"), "w") as f:
        f.write(yaml_content)
        
    print(f"\nSuccess! YOLOv26 Dataset generated at: {output_dir}")
    print("Images are cropped, masked, and ready for native YOLO training.")

# --- EXECUTE THE MODULE ---
SPLIT_JSON_PATH = 'MasterSplits/merged_dataset_splits.json'
build_yolo_bbox_dataset(SPLIT_JSON_PATH)

## YOLO Training

In [1]:
from ultralytics import YOLO
import os

def train_yolo_radar_model(data_yaml_path, epochs=50, batch_size=4, img_size=1024):
    """
    Initializes and trains the YOLO11 model on the pure bounding box radar dataset.
    Includes critical memory optimizations to prevent System RAM OOM crashes.
    """
    print("==================================================")
    print("INITIALIZING YOLO11 PIPELINE (LOW-RAM MODE)")
    print("==================================================")
    
    # 1. Verify the YAML file exists
    abs_yaml_path = os.path.abspath(data_yaml_path)
    if not os.path.exists(abs_yaml_path):
        raise FileNotFoundError(f"Cannot find {abs_yaml_path}. Did you run the dataset builder?")
        
    # 2. Load the YOLO26M architecture
    print("Loading YOLO26M base weights...")
    model = YOLO('yolo26m.pt')
    
    # 3. Launch the Training Loop
    print(f"Starting training for {epochs} epochs...")
    results = model.train(
        data=abs_yaml_path,
        epochs=epochs,
        imgsz=img_size,          
        batch=batch_size,        # Lowered to 4 or 8 to save both RAM and VRAM
        device=0,                
        
        # --- CRITICAL RAM OPTIMIZATIONS ---
        workers=0,               # FIXES RAM OOM: Forces data loading into the main thread. 
                                 # (If 0 is too slow, try 2. Default is usually 8, which crashes PCs).
        cache=False,             # Ensures images stay on disk and aren't pre-loaded into RAM.
        
        # --- Logging & Output ---
        project='RESAM_YOLO',    
        name='yolo11_radar_v1',  
        plots=True               
    )
    
    print("\n==================================================")
    print("YOLO11 TRAINING COMPLETE!")
    print("==================================================")
    print(f"Best model weights saved to: RESAM_YOLO/yolo11_radar_v1/weights/best.pt")
    
    return results

# --- EXECUTE THE TRAINING LOOP ---
YAML_PATH = "YOLO_Dataset/data.yaml"

# Start with a conservative batch_size=4 for 1024x1024 images
train_yolo_radar_model(
    data_yaml_path=YAML_PATH, 
    epochs=50, 
    batch_size=4, 
    img_size=1024
)

INITIALIZING YOLO11 PIPELINE (LOW-RAM MODE)
Loading YOLO11s base weights...
Starting training for 50 epochs...
Ultralytics 8.4.46 🚀 Python-3.12.3 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 8087MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/salvatorecapuozzo/RESAM/YOLO_Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt

INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.8). Upgrade using: pip install --upgrade albumentations


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 162.6±56.0 MB/s, size: 51.2 KB)
val: Scanning /home/salvatorecapuozzo/RESAM/YOLO_Dataset/labels/val.cache... 73 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 73/73 10.6Mit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001, momentum=0.9) with parameter groups 124 weight(decay=0.0), 136 weight(decay=0.0005), 136 bias(decay=0.0)
Plotting labels to /home/salvatorecapuozzo/RESAM/runs/detect/RESAM_YOLO/yolo11_radar_v1/labels.jpg... 
Image sizes 1024 train, 1024 val
Using 0 dataloader workers
Logging results to /home/salvatorecapuozzo/RESAM/runs/detect/RESAM_YOLO/yolo11_radar_v1
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss  

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d0dbdda1be0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

## YOLO Inference

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. EVALUATION CONFIGURATION
# ==========================================
WEIGHTS_PATH = "runs/detect/RESAM_YOLO/yolo26_radar_v1/weights/best.pt"
TEST_IMAGES_DIR = "YOLO_Dataset/images/test"
TEST_LABELS_DIR = "YOLO_Dataset/labels/test"
OUTPUT_VIS_DIR = "Inference_Results_YOLO"

# YOLO uses 0-indexed classes
CLASSES = {0: 'Ship', 1: 'Noise', 2: 'Unknown TGT', 3: 'Coast/Port', 4: 'My Ship', 5: 'AIS Lost'}

# ==========================================
# 2. UTILITY FUNCTIONS
# ==========================================
def calculate_box_iou(box1, box2):
    """Calculates Intersection over Union (IoU) between two bounding boxes [xmin, ymin, xmax, ymax]."""
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    iou = intersection_area / float(box1_area + box2_area - intersection_area)
    return iou

def get_image_map50(gt_boxes, pred_boxes, iou_threshold=0.5):
    """
    Calculates exact per-image mAP50 (Mean Average Precision).
    Plots the PR curve and calculates the exact Area Under the Curve (AP) per class.
    """
    if not gt_boxes and not pred_boxes: return 1.0  # Perfectly predicted empty image
    if not gt_boxes or not pred_boxes: return 0.0   # Missed targets or hallucinated on empty

    aps = []
    present_classes = set([int(g[0]) for g in gt_boxes] + [int(p[0]) for p in pred_boxes])
    
    for c in present_classes:
        gts_c = [g[1] for g in gt_boxes if int(g[0]) == c]
        preds_c = [p for p in pred_boxes if int(p[0]) == c]
        
        if not gts_c and not preds_c: continue
        if not gts_c or not preds_c:
            aps.append(0.0)
            continue
            
        # Sort predictions by confidence descending
        preds_c.sort(key=lambda x: x[2], reverse=True)
        
        tps = np.zeros(len(preds_c))
        fps = np.zeros(len(preds_c))
        matched_gts = set()
        
        for i, p in enumerate(preds_c):
            p_box = p[1]
            best_iou = 0
            best_gt_idx = -1
            
            for j, g_box in enumerate(gts_c):
                if j in matched_gts: continue
                iou = calculate_box_iou(p_box, g_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = j
                    
            if best_iou >= iou_threshold:
                tps[i] = 1
                matched_gts.add(best_gt_idx)
            else:
                fps[i] = 1
                
        # Calculate Precision and Recall curve
        cum_tps = np.cumsum(tps)
        cum_fps = np.cumsum(fps)
        recalls = cum_tps / len(gts_c)
        precisions = cum_tps / (cum_tps + cum_fps)
        
        # All-point interpolation AP
        precisions = np.concatenate(([0.0], precisions, [0.0]))
        recalls = np.concatenate(([0.0], recalls, [1.0]))
        for i in range(len(precisions) - 1, 0, -1):
            precisions[i - 1] = np.maximum(precisions[i - 1], precisions[i])
        
        indices = np.where(recalls[1:] != recalls[:-1])[0]
        ap = np.sum((recalls[indices + 1] - recalls[indices]) * precisions[indices + 1])
        aps.append(ap)
        
    return np.mean(aps) if aps else 0.0

def draw_boxes(ax, image, boxes, title):
    """Draws bounding boxes on a matplotlib axis."""
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
    
    cmap = plt.get_cmap('nipy_spectral')
    norm = plt.Normalize(vmin=0, vmax=5)
    
    for box_data in boxes:
        cls_id = int(box_data[0])
        box = box_data[1]
        conf = box_data[2] if len(box_data) > 2 else None
        
        xmin, ymin, xmax, ymax = box
        width, height = xmax - xmin, ymax - ymin
        color = cmap(norm(cls_id))
        
        rect = patches.Rectangle((xmin, ymin), width, height, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        label = f"{CLASSES.get(cls_id, 'Unknown')}"
        if conf is not None: label += f" {conf:.2f}"
        
        # Label background and text
        ax.text(xmin, ymin - 5, label, color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', alpha=0.7, pad=2))

# ==========================================
# 3. MAIN INFERENCE AND VISUALIZATION LOOP
# ==========================================
def evaluate_yolo_bbox():
    os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)
    
    print("==================================================")
    print("1. RUNNING NATIVE YOLO GLOBAL METRICS")
    print("==================================================")
    model = YOLO(WEIGHTS_PATH)
    # This calculates official mAP metrics across the whole test set
    metrics = model.val(data="YOLO_Dataset/data.yaml", split="test", plots=False)
    print(f"\nOverall mAP50: {metrics.box.map50:.4f}")
    print(f"Overall mAP50-95: {metrics.box.map:.4f}")

    print("\n==================================================")
    print("2. GENERATING ALL PER-IMAGE PREDICTIONS")
    print("==================================================")
    
    image_files = [f for f in os.listdir(TEST_IMAGES_DIR) if f.endswith('.png')]
    
    # Legend Patches
    cmap = plt.get_cmap('nipy_spectral')
    norm = plt.Normalize(vmin=0, vmax=5)
    legend_patches = [patches.Patch(color=cmap(norm(cid)), label=name) for cid, name in CLASSES.items()]
    
    for img_name in tqdm(image_files, desc="Processing Test Set"):
        img_path = os.path.join(TEST_IMAGES_DIR, img_name)
        txt_path = os.path.join(TEST_LABELS_DIR, img_name.replace('.png', '.txt'))
        
        # Load Image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_h, img_w, _ = image.shape
        
        # --- Load Ground Truth ---
        gt_boxes = []
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        xc, yc, w, h = map(float, parts[1:])
                        xmin = (xc - w / 2) * img_w
                        ymin = (yc - h / 2) * img_h
                        xmax = (xc + w / 2) * img_w
                        ymax = (yc + h / 2) * img_h
                        gt_boxes.append((cls_id, [xmin, ymin, xmax, ymax]))
                        
        # --- Get Predictions ---
        results = model.predict(image, conf=0.25, verbose=False)[0]
        pred_boxes = []
        if len(results.boxes) > 0:
            xyxy = results.boxes.xyxy.cpu().numpy()
            confs = results.boxes.conf.cpu().numpy()
            clses = results.boxes.cls.cpu().numpy()
            
            for i in range(len(xyxy)):
                pred_boxes.append((int(clses[i]), xyxy[i].tolist(), confs[i]))
                
        # --- Score and Plot the Image ---
        img_map50 = get_image_map50(gt_boxes, pred_boxes)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f"{img_name} | Image mAP50: {img_map50:.4f}", fontsize=16, fontweight='bold')
        
        axes[0].imshow(image)
        axes[0].set_title("Original Radar")
        axes[0].axis('off')
        
        draw_boxes(axes[1], image, gt_boxes, "Ground Truth")
        draw_boxes(axes[2], image, pred_boxes, "YOLO26 Prediction")
        axes[2].legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.05, 1), title="Classes")
        
        plt.tight_layout()
        save_name = f"{OUTPUT_VIS_DIR}/{img_name}"
        plt.savefig(save_name, bbox_inches='tight', dpi=150)
        plt.close()

    print(f"\nDone! Saved {len(image_files)} visual comparisons to the '{OUTPUT_VIS_DIR}' folder.")

# EXECUTE
evaluate_yolo_bbox()

1. RUNNING NATIVE YOLO GLOBAL METRICS
Ultralytics 8.4.46 🚀 Python-3.12.3 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 8087MiB)
YOLO26m summary (fused): 132 layers, 20,354,078 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2178.5±492.1 MB/s, size: 41.8 KB)
val: Scanning /home/salvatorecapuozzo/RESAM/YOLO_Dataset/labels/test.cache... 87 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 87/87 22.8Mit/s 0.0s
val: /home/salvatorecapuozzo/RESAM/YOLO_Dataset/images/test/R_283282.png: ignoring corrupt image/label: Label class 52 exceeds dataset class count 6. Possible class labels are 0-5
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.5it/s 4.1s0.7ss
                   all         86        526      0.707      0.728      0.757      0.363
                  Ship         58         95      0.708      0.737      0.759      0.265
                 Noise         40        125 

Processing Test Set: 100%|██████████| 87/87 [01:03<00:00,  1.37it/s]


Done! Saved 87 visual comparisons to the 'Inference_Results_YOLO' folder.
